In [ ]:
pip install torch torchvision pillow numpy kagglehub

In [ ]:
import os
import io
import random
import numpy as np
from PIL import Image, ImageFilter
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import kagglehub


In [ ]:
# ---------------------------------------------------------
# 1. Random Augmentation Pipeline Matching Exact Specs
# ---------------------------------------------------------
class RandomRedistributionAugmentation:
    """
    Applies random transformations simulating real-world redistribution pipelines:
    - JPEG Compression: quality in [90, 70, 50, 30]
    - Gaussian Blur: kernel sigma in [0.5, 1.0, 2.0]
    - Resize: downscale (0.5x / 0.25x) then upscale back
    - Gaussian Noise: sigma in [0.02, 0.05, 0.10]
    - Color Jitter: brightness/contrast/saturation +/- 20%
    - Center Crop: 80% crop
    """
    def __init__(self, p=0.5):
        self.p = p

    def jpeg_compression(self, img: Image.Image) -> Image.Image:
        quality = random.choice([90, 70, 50, 30])
        output = io.BytesIO()
        img.save(output, format='JPEG', quality=quality)
        output.seek(0)
        return Image.open(output)

    def gaussian_blur(self, img: Image.Image) -> Image.Image:
        sigma = random.choice([0.5, 1.0, 2.0])
        return img.filter(ImageFilter.GaussianBlur(radius=sigma))

    def resize_down_up(self, img: Image.Image) -> Image.Image:
        scale = random.choice([0.5, 0.25])
        w, h = img.size
        low_w, low_h = max(1, int(w * scale)), max(1, int(h * scale))
        
        img_resized = img.resize((low_w, low_h), Image.Resampling.BILINEAR)
        return img_resized.resize((w, h), Image.Resampling.BILINEAR)

    def gaussian_noise(self, img: Image.Image) -> Image.Image:
        sigma = random.choice([0.02, 0.05, 0.10])
        img_np = np.array(img).astype(np.float32) / 255.0
        
        noise = np.random.normal(0, sigma, img_np.shape)
        noisy_img = np.clip(img_np + noise, 0.0, 1.0) * 255.0
        return Image.fromarray(noisy_img.astype(np.uint8))

    def color_jitter(self, img: Image.Image) -> Image.Image:
        jitter = T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2)
        return jitter(img)

    def center_crop(self, img: Image.Image) -> Image.Image:
        w, h = img.size
        crop_w, crop_h = int(w * 0.8), int(h * 0.8)
        return TF.center_crop(img, [crop_h, crop_w])

    def __call__(self, img: Image.Image) -> Image.Image:
        transforms = [
            self.jpeg_compression,
            self.gaussian_blur,
            self.resize_down_up,
            self.gaussian_noise,
            self.color_jitter,
            self.center_crop
        ]
        
        for transform in transforms:
            if random.random() < self.p:
                img = transform(img)
                
        return img



In [ ]:

# ---------------------------------------------------------
# 2. CIFAKE PyTorch Dataset Class
# ---------------------------------------------------------
class CIFAKEDataset(Dataset):
    def __init__(self, root_dir, split='train', p_aug=0.5):
        """
        CIFAKE folder structure downloaded by kagglehub:
        root_dir/
            train/
                REAL/
                FAKE/
            test/
                REAL/
                FAKE/
        """
        self.dataset_dir = os.path.join(root_dir, split)
        self.image_paths = []
        self.labels = []
        self.augmentor = RandomRedistributionAugmentation(p=p_aug)
        self.to_tensor = T.ToTensor()

        # Load REAL (label 0) and FAKE (label 1) images
        for label_name, label_val in [('REAL', 0), ('FAKE', 1)]:
            folder = os.path.join(self.dataset_dir, label_name)
            if os.path.exists(folder):
                for fname in os.listdir(folder):
                    if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                        self.image_paths.append(os.path.join(folder, fname))
                        self.labels.append(label_val)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        
        # Apply random augmentations
        augmented_image = self.augmentor(image)
        b
        # Convert output to PyTorch Tensor
        tensor_image = self.to_tensor(augmented_image)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        
        return tensor_image, label

In [ ]:
# ---------------------------------------------------------
# 3. Execution & Data Loading
# ---------------------------------------------------------
if __name__ == '__main__':
    # Download CIFAKE dataset via kagglehub
    dataset_path = kagglehub.dataset_download("birdy654/cifake-real-and-ai-generated-synthetic-images")
    print("Path to dataset files:", dataset_path)

    # Instantiate CIFAKE training set with random augmentations
    train_dataset = CIFAKEDataset(root_dir=dataset_path, split='train', p_aug=0.5)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)

    # Test loading one batch
    images, labels = next(iter(train_loader))
    print(f"Batch successfully loaded! Images shape: {images.shape}, Labels shape: {labels.shape}")